<a href="https://colab.research.google.com/github/pradeepkumar2772/rea/blob/main/notebooks/SeedVC_Voice_Changer_by_AIQUEST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎙️ SeedVC Voice Changer & Singing Cloner</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Google Colab T4 Edition - Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0;'>Zero-shot voice conversion for speech & songs | Keeps the backing music | Powered by SeedVC + BS-RoFormer</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-T4%20GPU-4285F4?style=for-the-badge&logo=google-colab&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

### What it does

**SeedVC** turns any voice into any other voice from a single short clip, with no training. Give it a **source** (a speech or a song) and a **target voice** (5-30 s of the voice you want), and it re-sings or re-speaks the source in that voice, keeping the words, timing and melody.

| Feature | Details |
|----|----|
| 🗣️ Speech v1 | 22 kHz model for talking: voice-overs, dubbing, character voices |
| 🎤 Singing v1 | 44 kHz model with RMVPE pitch tracking: song covers keep the melody, with **pitch shift** (±24 semitones) and auto key matching |
| 🌍 Speech v2 | ASTRAL content encoder; can also copy the target's **accent & emotion** |
| 🎼 Keep the music | BS-RoFormer splits vocals from the backing track; only the voice is converted, then the original music is mixed back in |
| 💾 Output | WAV in `/content/outputs` (Colab Files panel), plus the converted vocals on their own |

### ⚡ Why it is fast

| What | Why it helps on a T4 |
|----|----|
| **All models stay in VRAM** | The three engines and the separator need ~6 GB together, so nothing is offloaded to RAM or re-uploaded between conversions. An engine loads on first use, then stays |
| **Short voice prompt** | SeedVC fits target voice + source into one 30 s window. Using the densest 10 s of the target (adjustable) leaves 20 s of source per window: a long song needs ~4x fewer windows |
| **FP16 vocal separation** | RoFormer (trained with mixed precision) runs in FP16, 4 chunks per batch, and stays loaded |
| **No MP3 previews** | SeedVC's per-chunk MP3 encoding for streaming players is skipped; only the final WAV is written |
| **Lean install** | Only the packages SeedVC uses, with Wan2GP pinned to a tested commit, so an upstream change can't break the notebook |

### Quick Start
1. **Runtime → Change runtime type → T4 GPU**
2. Run **Step 1** (~2-3 min: code, packages and ~6 GB of weights), then **Step 2** (loads the models, then prints the links)
3. Open the **Cloudflare** link or the **Gradio** share link printed at the bottom of Step 2
4. Add a **source** and a **target voice** (upload or record), pick an engine, press **🎙️ Change Voice**
5. **Tips:** a clean target clip without music works best (or tick *Remove music from the target voice clip*). For songs use **Singing**, and **pitch shift** +12 / -12 to cross between male and female ranges. Fewer **steps** or **CFG 0** = faster

In [ ]:
#@title 📦 Step 1: Setup + Model Downloads
# Pinned Wan2GP code (SeedVC), only the packages SeedVC needs, and ~6 GB of weights for all three engines + the vocal separator.
import os
import sys
import time
import json
import shutil
import subprocess
import threading
import psutil
from concurrent.futures import ThreadPoolExecutor


def stop_old_engine():
    # An engine still running from an earlier Step 2 (or left behind by a session restart) holds the GPU and RAM.
    stale = []
    for p in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            args, name = p.info["cmdline"] or [], (p.info["name"] or "").lower()
            engine = "python" in name and any(os.path.basename(a) == "run_seedvc.py" for a in args[1:3])
            if p.pid != os.getpid() and (engine or name == "cloudflared" or name.startswith("frpc")):
                p.kill()
                stale.append(p)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    psutil.wait_procs(stale, timeout=20)
    if stale:
        print(f"🧹 Stopped {len(stale)} engine process(es) from an earlier run: GPU memory and RAM are free again")


stop_old_engine()
T_START = time.time()

# ---- Settings ----
WAN2GP_COMMIT = "f02a38275fa43057ea75814e8ef4368e5530dcf2"  # deepbeepmeep/Wan2GP, 6 Oct 2026 (tested)
REPO_DIR = "/content/Wan2GP"
MODELS_DIR = "/content/seedvc_models"

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"  # no second copy of the weights in the xet cache
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("=" * 70)
print("🎙️ SeedVC Voice Changer & Singing Cloner")
print("📺 Created by: AIQUEST Academy | YouTube & X: @AIQuestAcademy")
print("=" * 70)
ram = psutil.virtual_memory()
print(f"RAM: {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available")
try:
    gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                                  text=True).strip().splitlines()[0]
    print(f"✅ GPU: {gpu}")
except Exception:
    print("❌ No GPU found: Runtime → Change runtime type → T4 GPU, then run this cell again")


def fmt_time(seconds):
    seconds = int(seconds)
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"


def run_quiet(label, cmd):
    """Run a command with one live spinner line; show its output only if it fails."""
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    reader = threading.Thread(target=lambda: lines.extend(proc.stdout), daemon=True)
    reader.start()
    frames = "⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏"
    while proc.poll() is None:
        sys.stdout.write(f"\r  {frames[int(time.time() * 4) % 10]} {label} | {fmt_time(time.time() - t0)}".ljust(80))
        sys.stdout.flush()
        time.sleep(0.25)
    reader.join(timeout=5)
    sys.stdout.write("\r" + " " * 80 + "\r")
    if proc.returncode != 0:
        print(f"  ❌ {label} failed:\n" + "".join(lines[-40:]))
        raise SystemExit(f"❌ {label} failed (see above)")
    print(f"  ✅ {label} ({fmt_time(time.time() - t0)})")


# ---- 1. Wan2GP code (SeedVC lives in postprocessing/seedvc), pinned to a tested commit ----
print("\n🧩 Code and dependencies")
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "init", "-q", REPO_DIR], check=True)
if subprocess.run(["git", "-C", REPO_DIR, "remote", "get-url", "origin"], capture_output=True).returncode != 0:
    subprocess.run(["git", "-C", REPO_DIR, "remote", "add", "origin", "https://github.com/deepbeepmeep/Wan2GP.git"],
                   check=True)
run_quiet(f"Wan2GP @ {WAN2GP_COMMIT[:10]}", ["git", "-C", REPO_DIR, "fetch", "-q", "--depth", "1", "origin", WAN2GP_COMMIT])
subprocess.run(["git", "-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD"], check=True)  # also undoes stale edits

# ---- 2. Python packages: only what SeedVC + the RoFormer separator import (Colab's torch / torchaudio stay) ----
run_quiet("Python packages", [
    sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", "--timeout", "120", "--retries", "5",
    "transformers==4.54.0", "huggingface_hub[hf_xet]<1.0", "gradio==5.29.0", "librosa==0.11.0", "munch",
    "hydra-core", "omegaconf", "pydub", "audioop-lts; python_version >= '3.13'", "einops", "beartype",
    "rotary-embedding-torch", "soundfile", "PyYAML"])
# BS-RoFormer model code only: the package's own dependencies (onnxruntime, diffq, ...) are for other separators
run_quiet("RoFormer model code", [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--timeout", "120",
                                  "--retries", "5", "audio-separator==0.36.1"])

# ---- 3. Model weights (parallel downloads, one progress bar) ----
from huggingface_hub import HfApi, hf_hub_download

SEEDVC_REPO, ROFORMER_REPO = "DeepBeepMeep/LTX-2", "DeepBeepMeep/Wan2.1"
WHISPER_FILES = ["added_tokens.json", "config.json", "generation_config.json", "merges.txt", "model.safetensors",
                 "normalizer.json", "preprocessor_config.json", "special_tokens_map.json", "tokenizer.json",
                 "tokenizer_config.json", "vocab.json"]
FILES = [(SEEDVC_REPO, f"seed-vc/{f}") for f in (
            "DiT_seed_v2_uvit_whisper_small_wavenet_bigvgan_pruned.pth",          # Speech v1
            "config_dit_mel_seed_uvit_whisper_small_wavenet.yml",
            "DiT_seed_v2_uvit_whisper_base_f0_44k_bigvgan_pruned_ft_ema_v2.pth",  # Singing v1
            "config_dit_mel_seed_uvit_whisper_base_f0_44k.yml",
            "rmvpe.pt",                                                           # pitch tracker (singing)
            "campplus_cn_common.bin",                                             # speaker style encoder
            "v2/ar_base.pth", "v2/cfm_small.pth",                                 # Speech v2
            "bsq32/bsq32_light.pth", "bsq2048/bsq2048_light.pth")] + \
        [(SEEDVC_REPO, f"{d}/{f}") for d in ("bigvgan_v2_22khz_80band_256x", "bigvgan_v2_44khz_128band_512x")
         for f in ("config.json", "bigvgan_generator.pt")] + \
        [(SEEDVC_REPO, f"whisper-small/{f}") for f in WHISPER_FILES] + \
        [(SEEDVC_REPO, f"hubert-large-ll60k/{f}") for f in ("config.json", "preprocessor_config.json", "pytorch_model.bin")] + \
        [(ROFORMER_REPO, f"roformer/model_bs_roformer_ep_317_sdr_12.9755.{ext}") for ext in ("ckpt", "yaml")]

api = HfApi()
SIZES = {}
for repo in (SEEDVC_REPO, ROFORMER_REPO):
    for info in api.get_paths_info(repo, [f for r, f in FILES if r == repo]):
        SIZES[(repo, info.path)] = getattr(info, "size", 0) or 0


def fetch(item):
    repo, filename = item
    if os.path.exists(os.path.join(MODELS_DIR, filename)):
        return f"✓ {filename}"
    for attempt in range(1, 4):
        try:
            hf_hub_download(repo_id=repo, filename=filename, local_dir=MODELS_DIR)
            size = SIZES.get(item, 0)
            return f"✅ {filename}" + (f" ({size / 1e9:.2f} GB)" if size > 100e6 else "")
        except Exception as e:
            if attempt == 3:
                raise
            print(f"  ⚠️ {filename}: {e} - retrying ({attempt}/3)")
            time.sleep(5 * attempt)


def folder_bytes(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            p = os.path.join(root, f)
            if not os.path.islink(p):
                try:
                    total += os.path.getsize(p)
                except OSError:
                    pass
    return total


os.makedirs(MODELS_DIR, exist_ok=True)
todo = [item for item in FILES if not os.path.exists(os.path.join(MODELS_DIR, item[1]))]
need = sum(SIZES.get(item, 0) for item in todo)
free_gb = shutil.disk_usage(MODELS_DIR).free / 1024**3
print(f"\n🚀 Downloading {need / 1e9:.1f} GB of weights to {MODELS_DIR} ({free_gb:.0f} GB free)...")
if need / 1024**3 > free_gb - 2:
    raise SystemExit(f"❌ Not enough disk space: need {need / 1e9:.0f} GB, {free_gb:.0f} GB free.")
done = threading.Event()
print_lock = threading.Lock()
bar_shown = [False]


def say(message):  # a normal line printed above the live bar
    with print_lock:
        if bar_shown[0]:
            sys.stdout.write("\r" + " " * 100 + "\r")
            bar_shown[0] = False
        print(message, flush=True)


def monitor():
    # One live line: GB done / total, %, smoothed speed, ETA. Partial files count, so bytes on disk is the progress.
    base = folder_bytes(MODELS_DIR)
    total = max(need, 1.0)
    start, samples = time.time(), []
    while not done.wait(1.0):
        got = folder_bytes(MODELS_DIR) - base
        now = time.time()
        samples = [(t, b) for t, b in samples if now - t <= 10] + [(now, got)]
        speed = (samples[-1][1] - samples[0][1]) / max(samples[-1][0] - samples[0][0], 1e-6) if len(samples) > 1 else 0
        got = min(got, total)
        frac = got / total
        fill = int(frac * 28)
        eta = fmt_time(max(0.0, total - got) / speed) if speed > 1e6 else "--"
        line = (f"  ⬇️ [{'█' * fill}{'░' * (28 - fill)}] {got / 1e9:5.1f} / {total / 1e9:.1f} GB {frac * 100:3.0f}% | "
                f"{speed / 1e6:4.0f} MB/s | ETA {eta} | {fmt_time(now - start)}")
        with print_lock:
            sys.stdout.write("\r" + line.ljust(100))
            sys.stdout.flush()
            bar_shown[0] = True


threading.Thread(target=monitor, daemon=True).start()
t0 = time.time()
try:
    with ThreadPoolExecutor(max_workers=8) as pool:
        jobs = [pool.submit(fetch, item) for item in sorted(FILES, key=lambda it: -SIZES.get(it, 0))]
        for job in jobs:
            result = job.result()
            if "GB" in result:  # only the big files get a line
                say("  " + result)
finally:
    done.set()
    time.sleep(1.1)
say(f"  ✅ {need / 1e9:.1f} GB downloaded in {fmt_time(time.time() - t0)}"
    + (f" (avg {need / 1e6 / max(time.time() - t0, 1):.0f} MB/s)" if need else ""))
shutil.rmtree(os.path.join(MODELS_DIR, ".cache"), ignore_errors=True)

# Settings for Step 2's engine (saved to a file, so Step 2 can be re-run alone after a session restart)
SETTINGS = {"SEEDVC_WAN2GP": REPO_DIR, "SEEDVC_MODELS": MODELS_DIR}
os.environ.update(SETTINGS)
with open("/content/seedvc_settings.json", "w") as f:
    json.dump(SETTINGS, f)
print("\n" + "=" * 70)
print(f"✅ Step 1 done in {fmt_time(time.time() - T_START)} | disk free: {shutil.disk_usage(MODELS_DIR).free / 1024**3:.0f} GB")
print("=" * 70)

In [ ]:
#@title 🚀 Step 2: Start the Voice Changer
# Writes run_seedvc.py, then runs it: model load + warm-up (~1 min) → Cloudflare + Gradio links.
# The page shows a live progress bar; reloading it or a dropped connection does not stop the job.
# Stop ends the job within a step. The newest audio is also at <link>/aiq_audio
import codecs
import json
import os
import re
import subprocess
import sys
import time

RUN_SCRIPT_CODE = r'''"""SeedVC voice changer engine for Google Colab (T4 GPU) - AIQUEST Academy.

Step 2 of the notebook writes this file as run_seedvc.py and runs it. It drives SeedVC (Plachtaa/seed-vc, through the
Wan2GP code pinned in Step 1) with these speed changes:

* Every model stays on the GPU. The three SeedVC engines and the RoFormer vocal separator fit on a T4 together, so
  nothing is offloaded to RAM or copied back to the GPU between conversions. An engine loads the first time its mode
  is used and then stays.
* No MP3 previews. SeedVC encodes every 30 s chunk to MP3 (one ffmpeg run per chunk) for a streaming player that is
  not used here; only the full-quality WAV is kept.
* Short voice prompt. SeedVC packs the target voice and the source into one 30 s window, so a 25 s target clip leaves
  5 s of source per chunk. The target clip is cut to its densest 10 s of voice by default: a 3-minute song needs about
  4x fewer chunks, with nearly the same timbre.
* RoFormer demixing in FP16 (the model was trained with mixed precision), 4 chunks per batch, loaded once.
* SeedVC weights stay in FP32 with FP16 autocast: the same precision SeedVC itself uses.
"""
import contextlib
import gc
import glob
import importlib
import importlib.machinery
import importlib.util
import io
import json
import logging
import math
import os
import re
import subprocess
import sys
import tempfile
import threading
import time
import types
import urllib.request
import warnings

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)

WAN2GP_DIR = os.environ.get("SEEDVC_WAN2GP", "/content/Wan2GP")
MODELS_DIR = os.environ.get("SEEDVC_MODELS", "/content/seedvc_models")
WORK_DIR = os.environ.get("SEEDVC_WORK_DIR", "/content" if os.path.isdir("/content") else os.getcwd())
OUTPUT_DIR = os.path.join(WORK_DIR, "outputs")
SERVER_PORT = 7860
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ.setdefault("GRADIO_TEMP_DIR", os.path.join(OUTPUT_DIR, ".gradio_cache"))
if WAN2GP_DIR not in sys.path:
    sys.path.insert(0, WAN2GP_DIR)

import librosa
import numpy as np
import psutil
import soundfile as sf
import torch
import torch.nn.functional as F
import yaml

import gradio as gr


def _gb(n):
    return f"{n / 1024**3:.1f} GB"


def _fmt_secs(seconds):
    seconds = int(round(seconds))
    return f"{seconds // 60}m{seconds % 60:02d}s" if seconds >= 60 else f"{seconds}s"


class ConsoleBar:
    """One live progress line per phase, redrawn every second with a carriage return (the notebook cell overwrites it
    in place). A finished phase is replaced by a one-line summary. Step-based phases interpolate inside the current
    step from the previous step's duration, so the bar keeps moving and the ETA counts down between steps."""

    WIDTH, LINE = 24, 104

    def __init__(self, stream):
        self.out = stream
        self.lock = threading.RLock()
        self.phase = None
        self.visible = False
        self.text_pending = False  # another writer is in the middle of a line: don't draw over it
        threading.Thread(target=self._loop, daemon=True).start()

    def start(self, key, label, total=None, unit=""):
        with self.lock:
            self.finish()
            now = time.time()
            self.phase = dict(key=key, label=label, total=total, unit=unit, done=0, t0=now, t_done=now, dt=None)
            self._draw()

    def update(self, done, total=None):
        with self.lock:
            p = self.phase
            if p is None:
                return
            now = time.time()
            if total:
                p["total"] = total
            if done > p["done"]:
                p["dt"] = (now - p["t_done"]) / (done - p["done"])
                p["t_done"], p["done"] = now, done
            self._draw()

    def finish(self, note=""):
        with self.lock:
            p, self.phase = self.phase, None
            if p is None:
                return
            elapsed = time.time() - p["t0"]
            line = f"  ✅ {p['label']}"
            if p["total"]:
                line += f" {p['done']}/{p['total']} {p['unit']}"
            line += f" in {_fmt_secs(elapsed) if elapsed >= 1 else f'{elapsed:.1f}s'}"
            if p["unit"] == "steps" and p["done"]:
                line += f" ({elapsed / p['done']:.2f}s/step)"
            if note:
                line += f" {note}"
            self._clear()
            self.out.write(line + "\n")
            self.out.flush()

    def abort(self, why="failed"):
        """End the live line of a phase that did not complete (no ✅ summary)."""
        with self.lock:
            p, self.phase = self.phase, None
            if p is None:
                return
            self._clear()
            self.out.write(f"  ❌ {p['label']} {why} after {_fmt_secs(time.time() - p['t0'])}\n")
            self.out.flush()

    def _render(self):
        p, now = self.phase, time.time()
        elapsed = now - p["t0"]
        if not p["total"]:
            spinner = "⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏"[int(now * 4) % 10]
            return f"  {spinner} {p['label']} | {_fmt_secs(elapsed)}"
        progress = p["done"]
        if p["dt"] and p["done"] < p["total"]:
            progress += min(0.97, (now - p["t_done"]) / p["dt"])
        frac = min(1.0, progress / p["total"])
        fill = int(round(frac * self.WIDTH))
        line = (f"  ⏳ {p['label']:<18} [{'█' * fill}{'░' * (self.WIDTH - fill)}] {frac * 100:3.0f}% | "
                f"{p['done']}/{p['total']} {p['unit']}")
        if p["dt"]:
            eta = max(0.0, (p["total"] - p["done"]) * p["dt"] - (now - p["t_done"]))
            line += f" | ETA {_fmt_secs(eta)}"
        return line + f" | {_fmt_secs(elapsed)}"

    def _draw(self):
        if self.phase is None or self.text_pending:
            return
        self.out.write("\r" + self._render().ljust(self.LINE))
        self.out.flush()
        self.visible = True

    def _clear(self):
        if self.visible:
            self.out.write("\r" + " " * self.LINE + "\r")
            self.visible = False

    def _loop(self):
        while True:
            time.sleep(1.0)
            with self.lock:
                self._draw()


class _BarAwareStream:
    """stdout / stderr wrapper: ordinary log lines clear the live bar first and print above it."""

    def __init__(self, stream, bar):
        self._stream, self._bar = stream, bar

    def write(self, data):
        with self._bar.lock:
            if data:
                self._bar._clear()
                self._bar.text_pending = not data.endswith("\n")
            return self._stream.write(data)

    def flush(self):
        self._stream.flush()

    def __getattr__(self, name):
        return getattr(self._stream, name)


BAR = ConsoleBar(sys.stdout)
sys.stdout = _BarAwareStream(sys.stdout, BAR)
sys.stderr = _BarAwareStream(sys.stderr, BAR)

if not torch.cuda.is_available():
    raise SystemExit("❌ No GPU found. Runtime → Change runtime type → T4 GPU, then run all cells again.")
DEVICE = torch.device("cuda")
torch.backends.cuda.enable_flash_sdp(False)  # Turing (SM 7.5) has no flash kernel; SDPA uses memory-efficient attention
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
torch.set_grad_enabled(False)

_props = torch.cuda.get_device_properties(DEVICE) if DEVICE.type == "cuda" else None
GPU_NAME = _props.name if _props else "CPU"
print("=" * 70)
print("🎙️ SeedVC Voice Changer & Singing Cloner | AIQUEST Academy")
print(f"⚡ GPU: {GPU_NAME} ({_gb(_props.total_memory) if _props else '-'}) | "
      f"RAM: {_gb(psutil.virtual_memory().total)} ({_gb(psutil.virtual_memory().available)} free)")
print("=" * 70)
sys.stdout.flush()

# ==== SeedVC code from the pinned Wan2GP checkout ====
# shared/utils/__init__.py only re-exports diffusion schedulers (it imports diffusers). SeedVC needs none of them, so
# the package is registered without running that file.
_utils = types.ModuleType("shared.utils")
_utils.__path__ = [os.path.join(WAN2GP_DIR, "shared", "utils")]
_utils.__spec__ = importlib.machinery.ModuleSpec("shared.utils", None, is_package=True)
_utils.__spec__.submodule_search_locations = _utils.__path__
sys.modules["shared.utils"] = _utils
import shared

shared.utils = _utils
from shared.utils import files_locator as fl
from shared.utils import media_control

fl.set_checkpoints_paths([MODELS_DIR])
from postprocessing import seedvc
from postprocessing.seedvc import app_svc, app_vc

for _name in ("httpx", "gradio", "urllib3", "transformers", "numba", "matplotlib"):
    logging.getLogger(_name).setLevel(logging.ERROR)


class _NoMP3:
    """Stands in for pydub.AudioSegment inside SeedVC's chunk loop: the streaming MP3 preview is never used."""

    def __init__(self, *args, **kwargs):
        pass

    def export(self, *args, **kwargs):
        return io.BytesIO(b"")


for _app in (app_vc, app_svc):
    _app.AudioSegment = _NoMP3


class Cancelled(Exception):
    pass


CANCEL = threading.Event()


class StepControl:
    """Plays the part of Wan2GP's MediaControl: SeedVC's denoising loops report every step through it and check it
    between DiT layers, so Stop takes effect within a fraction of a step."""

    def __init__(self, on_step):
        self.gen = {}
        self.on_step = on_step

    def send_cmd(self, command, data=None):
        if command == "progress" and data:
            step, total = data[0]
            self.on_step(int(step), int(total))

    def checkpoint(self):
        if CANCEL.is_set():
            self.gen["abort"] = True


# ==== Models (all on the GPU, each loaded once) ====
MODE_SPEECH, MODE_SINGING, MODE_ACCENT = 1, 2, 3
MODE_LABELS = {
    "🗣️ Speech · v1 (22 kHz)": MODE_SPEECH,
    "🎤 Singing · v1 (44 kHz + pitch)": MODE_SINGING,
    "🌍 Speech + accent · v2": MODE_ACCENT,
}
MODE_NAMES = {MODE_SPEECH: "Speech v1", MODE_SINGING: "Singing v1", MODE_ACCENT: "Speech v2"}
MODE_DEFAULTS = {MODE_SPEECH: (25, 0.5), MODE_SINGING: (10, 0.7), MODE_ACCENT: (30, 0.7)}  # SeedVC's steps / CFG
V2_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32  # v2 autocast dtype (FP16 on the GPU)
CONVERTERS = {}
LOAD_LOCK = threading.Lock()


def _freeze(module):
    module.eval()
    for p in module.parameters():
        p.requires_grad_(False)


def get_converter(mode):
    with LOAD_LOCK:
        if mode in CONVERTERS:
            return CONVERTERS[mode]
        BAR.start(f"load{mode}", f"Loading SeedVC {MODE_NAMES[mode]}")
        try:
            with contextlib.redirect_stdout(io.StringIO()):  # SeedVC's own loading chatter
                converter = seedvc.get_model(dtype=torch.float16, mode=mode)
            if mode == MODE_ACCENT:
                converter.vc_wrapper.to(DEVICE)
                _freeze(converter.vc_wrapper)
                for module in converter.vc_wrapper.ar.modules():  # KV caches in the dtype the AR model runs in
                    if type(module).__name__ == "KVCache":
                        module.to(dtype=V2_DTYPE)
            for module in converter.pipe_modules().values():
                module.to(DEVICE)
                _freeze(module)
            if mode != MODE_ACCENT:
                converter._app_vc.device = DEVICE
                if getattr(converter, "f0_extractor", None) is not None:
                    converter.f0_extractor.device = DEVICE
        except BaseException:
            BAR.abort()
            raise
        gc.collect()
        BAR.finish(f"| VRAM in use: {_gb(torch.cuda.memory_allocated(DEVICE))}" if DEVICE.type == "cuda" else "")
        CONVERTERS[mode] = converter
        return converter


def _roformer_classes():
    """BS-RoFormer model code from the audio-separator package, imported on its own: the package's __init__ pulls in
    onnxruntime and a dozen other libraries this model never uses."""
    spec = importlib.util.find_spec("audio_separator")
    root = os.path.join(list(spec.submodule_search_locations)[0], "separator", "uvr_lib_v5", "roformer")
    pkg = types.ModuleType("aiq_roformer")
    pkg.__path__ = [root]
    pkg.__spec__ = importlib.machinery.ModuleSpec("aiq_roformer", None, is_package=True)
    pkg.__spec__.submodule_search_locations = pkg.__path__
    sys.modules["aiq_roformer"] = pkg
    return importlib.import_module("aiq_roformer.bs_roformer").BSRoformer, importlib.import_module("aiq_roformer.attend").Attend


class RoformerDemixer:
    """Vocal / backing-track split with the BS-RoFormer checkpoint Wan2GP uses, run like audio-separator does
    (8 s chunks, Hamming-weighted overlap-add) but in FP16 and several chunks per batch."""

    SR = 44100
    NAME = "model_bs_roformer_ep_317_sdr_12.9755"

    def __init__(self, folder, batch=4):
        with open(os.path.join(folder, self.NAME + ".yaml"), encoding="utf-8") as f:
            cfg = yaml.load(f, Loader=yaml.FullLoader)  # the config uses !!python/tuple
        BSRoformer, Attend = _roformer_classes()
        # PyTorch picks the fastest attention kernel itself (memory-efficient on a T4).
        Attend.flash_attn = lambda self, q, k, v: F.scaled_dot_product_attention(q, k, v)
        self.model = BSRoformer(**cfg["model"])
        self.model.load_state_dict(torch.load(os.path.join(folder, self.NAME + ".ckpt"), map_location="cpu",
                                              weights_only=True))
        self.model.to(DEVICE)
        _freeze(self.model)
        for estimator in self.model.mask_estimators:  # the complex mask multiply + iSTFT run in FP32
            estimator.register_forward_hook(lambda _module, _inputs, output: output.float())
        self.chunk = cfg["audio"]["hop_length"] * (cfg["inference"]["dim_t"] - 1)  # 352800 samples = 8 s
        self.batch = batch

    def separate(self, wave, sr, overlap=1, on_progress=None):
        """wave: (channels, samples) float32 at any rate. Returns (vocals, backing), (2, samples) at 44.1 kHz."""
        mix = to_stereo(resample(wave, sr, self.SR))
        peak = float(np.abs(mix).max()) if mix.size else 0.0
        if peak > 0.9:  # audio-separator's input normalisation
            mix = mix * (0.9 / peak)
        total, size = mix.shape[1], self.chunk
        padded = mix if total >= size else np.pad(mix, ((0, 0), (0, size - total)))
        length = padded.shape[1]
        starts = sorted({min(s, length - size) for s in range(0, length, max(1, size // int(overlap)))})
        source = torch.from_numpy(np.ascontiguousarray(padded))
        window = torch.hamming_window(size, periodic=False)
        acc = torch.zeros(2, length)
        weight = torch.zeros(length)
        for i in range(0, len(starts), self.batch):
            if CANCEL.is_set():
                raise Cancelled()
            group = starts[i:i + self.batch]
            parts = torch.stack([source[:, s:s + size] for s in group]).to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                pred = self.model(parts)
            pred = pred.float().cpu()[..., :size]
            for j, s in enumerate(group):
                acc[:, s:s + pred.shape[-1]] += pred[j] * window[:pred.shape[-1]]
                weight[s:s + pred.shape[-1]] += window[:pred.shape[-1]]
            if on_progress:
                on_progress(min(i + self.batch, len(starts)), len(starts))
        vocals = (acc / weight.clamp(min=1e-10))[:, :total].numpy()
        return vocals, mix - vocals  # vocals + backing = the mix, exactly


# ==== Audio helpers ====
def read_audio(path):
    """Any audio (or video) file → (channels, samples) float32 and its sample rate."""
    try:
        data, sr = sf.read(path, dtype="float32", always_2d=True)
    except Exception:  # m4a / aac / webm / mp4...: decode with ffmpeg
        with tempfile.TemporaryDirectory() as tmp:
            wav = os.path.join(tmp, "decoded.wav")
            subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", path, "-vn", "-c:a", "pcm_f32le", wav], check=True,
                           capture_output=True)
            data, sr = sf.read(wav, dtype="float32", always_2d=True)
    if data.shape[0] == 0:
        raise ValueError(f"{os.path.basename(path)} has no audio")
    return np.ascontiguousarray(data.T), int(sr)


def resample(wave, sr, target):
    if int(sr) == int(target):
        return np.asarray(wave, dtype=np.float32)
    return librosa.resample(np.asarray(wave, dtype=np.float32), orig_sr=int(sr), target_sr=int(target), axis=-1)


def to_mono(wave):
    return wave.mean(axis=0) if wave.ndim == 2 else wave


def to_stereo(wave):
    if wave.ndim == 1:
        wave = wave[None]
    return np.repeat(wave[:1], 2, axis=0) if wave.shape[0] == 1 else wave[:2]


def _frames(mono, hop):
    n = len(mono) // hop * hop
    return mono[:n].reshape(-1, hop)


def voiced_rms(wave, hop=1024):
    """RMS over the frames that carry sound (louder than 10% of the mean level): silence does not drag it down."""
    mono = to_mono(np.asarray(wave, dtype=np.float32))
    frames = _frames(mono, hop)
    if frames.size == 0:
        return float(np.sqrt(np.mean(mono ** 2))) if mono.size else 0.0
    level = np.abs(frames).mean(axis=1)
    voiced = frames[level > 0.1 * level.mean()]
    return float(np.sqrt(np.mean(voiced ** 2))) if voiced.size else 0.0


def match_loudness(wave, reference):
    a, b = voiced_rms(wave), voiced_rms(reference)
    if a <= 1e-8 or b <= 1e-8:
        return wave
    out = wave * (b / a)
    peak = float(np.abs(out).max())
    return out / peak if peak > 1.0 else out


def pick_reference(mono, sr, max_seconds):
    """Trim the silence, then keep the max_seconds stretch with the most voice in it."""
    trimmed, _ = librosa.effects.trim(mono, top_db=40)
    if trimmed.size >= sr:
        mono = trimmed
    keep = int(max_seconds * sr)
    if mono.shape[-1] <= keep:
        return mono
    hop = sr // 20  # 50 ms
    level = np.sqrt((_frames(mono, hop) ** 2).mean(axis=1))
    voiced = np.concatenate([[0.0], np.cumsum(level > 0.1 * level.mean())])
    span = keep // hop
    best = int(np.argmax(voiced[span:] - voiced[:-span])) if len(voiced) > span else 0
    return mono[best * hop: best * hop + keep]


# ==== Voice conversion ====
def _window_chunks(mode, converter, source_s, ref_s):
    """How many 30 s windows SeedVC will run (for the progress bar)."""
    if mode == MODE_ACCENT:
        w = converter.vc_wrapper
        sr, hop, ctx, overlap = w.sr, w.hop_size, w.dit_max_context_len, w.overlap_frame_len
    else:
        app = converter._app_vc
        sr, hop, ctx, overlap = app.sr, app.hop_length, 30, app.overlap_frame_len
    window = sr // hop * ctx - int(ref_s * sr / hop)
    frames = int(source_s * sr / hop)
    if frames <= window:
        return 1
    return 1 + math.ceil((frames - window) / max(1, window - overlap))


def seedvc_convert(mode, src_mono, src_sr, ref_mono, ref_sr, steps, cfg, pitch_shift, auto_f0, convert_style):
    """Mono source + mono target voice → (sample rate, converted mono float32)."""
    converter = get_converter(mode)
    if mode == MODE_ACCENT:
        generator = converter.vc_wrapper.convert_voice_arrays(
            source_wave=src_mono, target_wave=ref_mono, source_sr=src_sr, target_sr=ref_sr, diffusion_steps=steps,
            length_adjust=1.0, intelligebility_cfg_rate=cfg, similarity_cfg_rate=cfg, top_p=0.9, temperature=1.0,
            repetition_penalty=1.0, convert_style=convert_style, anonymization_only=False, device=DEVICE,
            dtype=V2_DTYPE)
        out_sr, out = seedvc._consume_generator_return(generator)
    else:
        sr = converter.sample_rate
        with tempfile.TemporaryDirectory() as tmp:
            src_path, ref_path = os.path.join(tmp, "source.wav"), os.path.join(tmp, "target.wav")
            sf.write(src_path, resample(src_mono, src_sr, sr), sr, subtype="FLOAT")
            sf.write(ref_path, resample(ref_mono, ref_sr, sr), sr, subtype="FLOAT")
            args = (src_path, ref_path, steps, 1.0, cfg) + ((auto_f0, pitch_shift) if mode == MODE_SINGING else ())
            result = None
            for item in converter._app_vc.voice_conversion(*args):
                if isinstance(item, tuple) and len(item) == 2 and item[1] is not None:
                    result = item[1]
        if result is None:
            raise RuntimeError("SeedVC produced no audio")
        out_sr, out = result
    return int(out_sr), seedvc._normalise_output(np.asarray(out, dtype=np.float32).reshape(-1))


# ==== Live progress (polled by the page over plain HTTP; the tunnels can hold back Gradio's event stream) ====
PROGRESS = {"active": False, "fraction": 0.0, "text": "", "started": 0.0, "finished": 0.0, "audio": None, "message": ""}


def report(fraction, text):
    PROGRESS.update(active=True, fraction=float(fraction), text=text)


def convert_job(source_path, target_path, mode, demix, clean_target, steps, cfg, ref_seconds, pitch_shift, auto_f0,
                convert_style, hq_demix):
    """Demix (optional) → SeedVC → loudness match → remix → save. Returns (song wav, vocals wav or None, status)."""
    t0 = time.time()
    steps, cfg = int(steps), float(cfg)
    source, source_sr = read_audio(source_path)
    target, target_sr = read_audio(target_path)
    source_s = source.shape[1] / source_sr
    print(f"\n🎙️ Job {JOB['id']} | {MODE_NAMES[mode]} | {source_s:.1f} s source | {steps} steps | CFG {cfg:g} | "
          f"target voice ≤ {ref_seconds:g} s | keep music: {'on' if demix else 'off'}")
    sys.stdout.flush()

    backing = None
    convert_from, convert_to = 0.02, 0.97
    target = to_mono(target)
    if clean_target:  # only the densest 30 s of the target clip is worth separating
        target = pick_reference(target, target_sr, 30.0)
    if demix or clean_target:
        overlap = 2 if hq_demix else 1
        span = 0.25 if demix else 0.05
        jobs = ([("source", source, source_sr)] if demix else []) + ([("target", target, target_sr)] if clean_target else [])
        for idx, (what, wave, sr) in enumerate(jobs):
            label = "Separating the music" if what == "source" else "Cleaning the target voice"
            base = span * idx / len(jobs)

            def on_demix(done, total, base=base, label=label):
                BAR.update(done, total)
                report(base + span / len(jobs) * done / total, f"{label} · {done}/{total} chunks")

            report(base, f"{label}...")
            BAR.start(f"demix-{what}", f"Demix ({what})", unit="chunks")
            try:
                vocals, rest = DEMIX.separate(wave, sr, overlap=overlap, on_progress=on_demix)
            except BaseException:
                BAR.abort("stopped" if CANCEL.is_set() else "failed")
                raise
            BAR.finish()
            if what == "source":
                source, source_sr, backing = vocals, RoformerDemixer.SR, rest
            else:
                target, target_sr = to_mono(vocals), RoformerDemixer.SR
        convert_from = span

    src_mono = to_mono(source)
    ref_mono = pick_reference(target, target_sr, float(ref_seconds))
    ref_mono = match_loudness(ref_mono, src_mono)  # SeedVC hears the target voice at the source's level
    if mode not in CONVERTERS:
        report(convert_from, f"Loading the {MODE_NAMES[mode]} engine (first use only)...")
    converter = get_converter(mode)
    chunks = _window_chunks(mode, converter, src_mono.shape[-1] / source_sr, ref_mono.shape[-1] / target_sr)
    state = {"chunk": 0, "chunks": chunks}

    def on_step(step, total):
        if step == 0:
            state["chunk"] += 1
            state["chunks"] = max(state["chunks"], state["chunk"])
        done = (state["chunk"] - 1) * total + step
        BAR.update(done, state["chunks"] * total)
        report(convert_from + (convert_to - convert_from) * done / (state["chunks"] * total),
               f"Converting the voice · window {state['chunk']}/{state['chunks']} · step {step}/{total}")

    report(convert_from, "Converting the voice...")
    BAR.start("convert", f"SeedVC {MODE_NAMES[mode]}", total=chunks * steps, unit="steps")
    token = media_control._control.set(StepControl(on_step))
    try:
        with torch.inference_mode():
            out_sr, voice = seedvc_convert(mode, src_mono, source_sr, ref_mono, target_sr, steps, cfg,
                                           int(pitch_shift), bool(auto_f0), bool(convert_style))
    except media_control.MediaProcessingAborted:
        BAR.abort("stopped")
        raise Cancelled()
    except BaseException:
        BAR.abort("stopped" if CANCEL.is_set() else "failed")
        raise
    finally:
        media_control._control.reset(token)
    BAR.finish(f"| {state['chunk']} window(s)")

    report(0.98, "Mixing + saving...")
    voice = match_loudness(voice, src_mono)  # same loudness as the original voice
    stamp = time.strftime("%Y%m%d_%H%M%S")
    tag = {MODE_SPEECH: "speech", MODE_SINGING: "singing", MODE_ACCENT: "v2"}[mode]
    vocals_path = None
    if backing is not None:
        voice = resample(voice, out_sr, RoformerDemixer.SR)
        out_sr = RoformerDemixer.SR
        n = backing.shape[1]
        voice = np.pad(voice, (0, max(0, n - voice.shape[0])))[:n]
        vocals_path = os.path.join(OUTPUT_DIR, f"seedvc_{tag}_{stamp}_vocals.wav")
        sf.write(vocals_path, voice, out_sr, subtype="PCM_16")
        song = backing + voice[None, :]
    else:
        song = voice[None, :]
    peak = float(np.abs(song).max()) if song.size else 0.0
    if peak > 0.99:  # scale instead of clipping
        song = song * (0.99 / peak)
    song_path = os.path.join(OUTPUT_DIR, f"seedvc_{tag}_{stamp}.wav")
    sf.write(song_path, song.T, out_sr, subtype="PCM_16")
    elapsed = time.time() - t0
    speed = source_s / max(elapsed, 1e-6)
    print(f"  💾 {source_s:.1f} s of audio in {_fmt_secs(elapsed)} ({speed:.1f}x real time) → {song_path}")
    sys.stdout.flush()
    status = (f"✅ {source_s:.1f} s converted in {elapsed:.1f} s ({speed:.1f}x real time) · {MODE_NAMES[mode]} · "
              f"{steps} steps · {state['chunk']} window(s)\n💾 {song_path}")
    return song_path, vocals_path, status


# ==== Startup: vocal separator + the speech engine, warmed up ====
T_LOAD = time.time()
BAR.start("demixer", "Loading RoFormer demixer")
DEMIX = RoformerDemixer(os.path.join(MODELS_DIR, "roformer"))
BAR.finish()
get_converter(MODE_SPEECH)
BAR.start("warmup", "Warm-up run")
try:
    with torch.inference_mode():
        _noise = (np.random.default_rng(0).standard_normal(22050 * 2) * 0.05).astype(np.float32)
        DEMIX.separate(_noise[None], 22050)
        seedvc_convert(MODE_SPEECH, _noise, 22050, _noise, 22050, 2, 0.5, 0, True, False)
    BAR.finish()
except Exception as e:  # only a speed-up for the first job
    BAR.abort(f"skipped ({e})")
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
print(f"✅ Ready in {_fmt_secs(time.time() - T_LOAD)} | VRAM in use: "
      f"{_gb(torch.cuda.memory_allocated(DEVICE)) if DEVICE.type == 'cuda' else '-'}")
sys.stdout.flush()


# ==== Conversion as a background job ====
# The Convert button only starts a job and returns. The page asks for the job's state every second with short,
# independent requests (queue=False: plain HTTP, no event stream), so a dropped tunnel connection or a reloaded page
# never loses the result.
JOB = {"id": 0, "done": 0, "running": False, "audio": None, "vocals": None, "status": "Ready."}


def _job_worker(job_id, args):
    try:
        with torch.inference_mode():  # grad mode is per thread: this one starts with it on
            audio, vocals, status = convert_job(*args)
    except Cancelled:
        audio, vocals, status = None, None, "⏹️ Stopped."
    except Exception as e:
        import traceback
        traceback.print_exc()
        audio, vocals, status = None, None, f"❌ Error: {e}"
    finally:
        CANCEL.clear()
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    JOB.update(running=False, done=job_id, audio=audio, vocals=vocals, status=status)
    PROGRESS.update(active=False, finished=time.time(), fraction=1.0 if audio else PROGRESS["fraction"],
                    audio=os.path.basename(audio) if audio else None, message=status.splitlines()[0])


def start_conversion(source_audio, target_voice, mode_label, demix, clean_target, steps, cfg, ref_seconds, pitch_shift,
                     auto_f0, convert_style, hq_demix):
    if JOB["running"]:
        return gr.skip(), gr.skip(), "⏳ A conversion is already running. The result appears here when it is ready."
    if not source_audio or not os.path.exists(str(source_audio)):
        return gr.skip(), gr.skip(), "❌ Add the source audio (the speech or song to convert)."
    if not target_voice or not os.path.exists(str(target_voice)):
        return gr.skip(), gr.skip(), "❌ Add the target voice (a clip of the voice you want)."
    mode = MODE_LABELS.get(mode_label, MODE_SPEECH)
    args = (source_audio, target_voice, mode, bool(demix), bool(clean_target), steps, cfg, ref_seconds, pitch_shift,
            auto_f0, convert_style, hq_demix)
    JOB.update(id=JOB["id"] + 1, running=True)
    PROGRESS.update(active=True, fraction=0.0, text="Starting...", started=time.time(), finished=0.0, audio=None,
                    message="")
    threading.Thread(target=_job_worker, args=(JOB["id"], args), daemon=True).start()
    return None, None, "🚀 Converting... the progress bar above updates live, and the audio appears here when it is ready."


def _progress_card(done=False):
    p = PROGRESS
    end = p["finished"] or time.time()
    elapsed = end - p["started"] if p["started"] else 0.0
    frac = max(0.0, min(1.0, p["fraction"]))
    if done:
        ok = bool(p["audio"])
        color = "linear-gradient(135deg,#10b981,#059669)" if ok else "linear-gradient(135deg,#ef4444,#b91c1c)"
        link = (' · <a href="aiq_audio?t=%d" target="_blank" style="color:#667eea;font-weight:600">▶ open / download</a>'
                % int(time.time())) if ok else ""
        text = (p["message"] or ("Done" if ok else "Stopped")) + link
        frac = 1.0 if ok else frac
    else:
        color = "linear-gradient(135deg,#667eea 0%,#764ba2 100%)"
        eta = ""
        if 0.05 < frac < 0.97 and elapsed > 3:
            eta = f" · ETA {_fmt_secs(elapsed * (1 - frac) / frac)}"
        text = f"⏳ {p['text'] or 'Working...'} · {_fmt_secs(elapsed)}{eta}"
    return (f'<div class="aiq-progress"><div class="aiq-progress-top"><span>{text}</span>'
            f'<span class="aiq-pct">{frac * 100:.0f}%</span></div><div class="aiq-track">'
            f'<div class="aiq-fill" style="width:{frac * 100:.1f}%;background:{color}"></div></div></div>')


IDLE_CARD = ('<div class="aiq-progress"><div class="aiq-progress-top"><span>💤 Ready. Add a source and a target voice, '
             'then press Change Voice.</span><span class="aiq-pct"></span></div><div class="aiq-track">'
             '<div class="aiq-fill" style="width:0%"></div></div></div>')


def poll_job(seen):
    """Every second: live progress while a job runs; the result once per page when it is done."""
    if JOB["running"]:
        return _progress_card(), gr.skip(), gr.skip(), gr.skip(), seen
    if JOB["done"] and JOB["done"] != seen:
        return _progress_card(done=True), JOB["audio"], JOB["vocals"], JOB["status"], JOB["done"]
    return gr.skip(), gr.skip(), gr.skip(), gr.skip(), seen


def stop_conversion():
    if not JOB["running"]:
        return "Nothing is running."
    CANCEL.set()
    return "⏹️ Stopping..."


def _latest_audio():
    files = [f for f in glob.glob(os.path.join(OUTPUT_DIR, "seedvc_*.wav")) if not f.endswith("_vocals.wav")]
    return max(files, key=os.path.getmtime) if files else None


def load_latest_audio():
    path = _latest_audio()
    if not path:
        return None, None, f"No converted audio in {OUTPUT_DIR} yet."
    vocals = path[:-4] + "_vocals.wav"
    return path, vocals if os.path.exists(vocals) else None, f"📁 Latest audio: {path}"


def on_mode_change(mode_label):
    mode = MODE_LABELS.get(mode_label, MODE_SPEECH)
    steps, cfg = MODE_DEFAULTS[mode]
    singing = mode == MODE_SINGING
    return (gr.update(value=steps), gr.update(value=cfg), gr.update(visible=singing), gr.update(visible=singing),
            gr.update(visible=mode == MODE_ACCENT))


# ==== Gradio UI (AIQUEST Academy branding) ====
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.status-line { text-align: center; color: #6b7280; font-size: 13px; margin: -8px 0 12px 0; }
.brand-footer { text-align: center; color: #6b7280; font-size: 12px; margin-top: 24px; }
.aiq-progress { padding: 12px 14px; border-radius: 12px; background: rgba(102,126,234,0.07); border: 1px solid rgba(102,126,234,0.25); }
.aiq-progress-top { display: flex; justify-content: space-between; gap: 12px; font-size: 13px; margin-bottom: 8px; }
.aiq-pct { font-weight: 700; color: #764ba2; }
.aiq-track { height: 10px; background: rgba(118,75,162,0.15); border-radius: 6px; overflow: hidden; }
.aiq-fill { height: 100%; border-radius: 6px; transition: width .8s ease; }
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">🎙️ SeedVC Voice Changer & Singing Cloner</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; Google Colab T4 Edition</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""
STATUS_HTML = (f'<div class="status-line">⚡ {GPU_NAME} &nbsp;|&nbsp; all models kept in VRAM &nbsp;|&nbsp; '
               f'FP16 RoFormer vocal separation &nbsp;|&nbsp; zero-shot: no training needed</div>')
FOOTER_HTML = ('<div class="brand-footer">⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · '
               '© All rights reserved</div>')
_DEFAULT_MODE = next(iter(MODE_LABELS))

print("🖥️ Building the UI...")
sys.stdout.flush()
with gr.Blocks(theme=gr.themes.Soft(), css=CSS, title="SeedVC Voice Changer | AIQUEST Academy",
               analytics_enabled=False) as demo:
    gr.HTML(BRAND_HTML)
    gr.HTML(STATUS_HTML)
    gr.Markdown("💡 **How it works:** upload (or record) the **source**, the speech or song you want to change, and "
                "a **target voice**, a clean 5-30 s clip of the new voice. With **Keep the music** on, the vocals are "
                "separated first, converted, and mixed back over the original backing track.")
    with gr.Row():
        with gr.Column(scale=1):
            source_audio = gr.Audio(label="🎵 Source audio (speech or song)", type="filepath")
            target_voice = gr.Audio(label="👤 Target voice (5-30 s of clean voice)", type="filepath")
            mode_radio = gr.Radio(list(MODE_LABELS), value=_DEFAULT_MODE, label="🎛️ Engine",
                                  info="Singing keeps the melody (pitch tracking); v2 can also copy the accent")
            demix_box = gr.Checkbox(value=True, label="🎼 Keep the music (separate vocals, convert, remix)")
            clean_box = gr.Checkbox(value=False, label="🧼 Remove music from the target voice clip too")
        with gr.Column(scale=1):
            with gr.Accordion("⚙️ Quality / speed", open=True):
                steps_slider = gr.Slider(4, 100, value=MODE_DEFAULTS[MODE_SPEECH][0], step=1, label="⚡ Diffusion steps",
                                         info="Fewer = faster. Speech 25 · singing 10-30 · v2 30")
                cfg_slider = gr.Slider(0.0, 1.5, value=MODE_DEFAULTS[MODE_SPEECH][1], step=0.05,
                                       label="🎯 CFG rate (similarity)", info="0 = fastest (one pass per step)")
                ref_slider = gr.Slider(3, 25, value=10, step=1, label="🎙️ Target voice length used (s)",
                                       info="Longer = closer timbre, but slower on long sources")
                pitch_slider = gr.Slider(-24, 24, value=0, step=1, label="🎹 Pitch shift (semitones)",
                                         info="e.g. +12 male → female, -12 female → male", visible=False)
                auto_f0_box = gr.Checkbox(value=True, label="🎚️ Auto-match the target voice's key", visible=False)
                style_box = gr.Checkbox(value=False, label="🌍 Also copy accent & emotion (slower)", visible=False)
                hq_box = gr.Checkbox(value=False, label="💎 High-quality separation (2x overlap, slower)")
            with gr.Row():
                gen_btn = gr.Button("🎙️ Change Voice", variant="primary", size="lg")
                stop_btn = gr.Button("🛑 Stop", size="lg", elem_id="stop-btn")
                clear_btn = gr.Button("🗑️ Clear", size="lg", elem_id="clear-btn")
            progress_html = gr.HTML(IDLE_CARD)
            audio_out = gr.Audio(label="🔊 Converted audio", type="filepath", interactive=False)
            vocals_out = gr.Audio(label="🎤 Converted vocals only (when the music is kept)", type="filepath",
                                  interactive=False)
            status_box = gr.Textbox(label="ℹ️ Status", lines=2, interactive=False)
            latest_btn = gr.Button("📂 Load latest audio (if the page lost its connection)")
    seen_job = gr.State(0)
    poll_timer = gr.Timer(1.0)
    gr.HTML(FOOTER_HTML)

    mode_radio.change(on_mode_change, inputs=mode_radio,
                      outputs=[steps_slider, cfg_slider, pitch_slider, auto_f0_box, style_box], queue=False,
                      show_progress="hidden")
    gen_btn.click(start_conversion,
                  inputs=[source_audio, target_voice, mode_radio, demix_box, clean_box, steps_slider, cfg_slider,
                          ref_slider, pitch_slider, auto_f0_box, style_box, hq_box],
                  outputs=[audio_out, vocals_out, status_box], queue=False)
    poll_timer.tick(poll_job, inputs=seen_job, outputs=[progress_html, audio_out, vocals_out, status_box, seen_job],
                    queue=False, show_progress="hidden")
    stop_btn.click(stop_conversion, outputs=status_box, queue=False)
    latest_btn.click(load_latest_audio, outputs=[audio_out, vocals_out, status_box], queue=False)
    clear_btn.click(lambda: (None, None, None, None, "Ready.", IDLE_CARD),
                    outputs=[source_audio, target_voice, audio_out, vocals_out, status_box, progress_html],
                    queue=False)


# ==== Cloudflare quick tunnel: a second public link in case the Gradio share link fails ====
# cloudflared tries QUIC (UDP) first, which some networks block: hence --protocol http2. A browser that asks for the
# random *.trycloudflare.com name before it exists caches the "not found" answer for minutes, so the link is only shown
# as ready once cloudflared has registered AND the name resolves.
def _dns_published(host):
    """Ask Cloudflare's DNS-over-HTTPS directly (no local resolver caches a negative answer). True / False / None."""
    try:
        req = urllib.request.Request(f"https://cloudflare-dns.com/dns-query?name={host}&type=A",
                                     headers={"accept": "application/dns-json"})
        return bool(json.loads(urllib.request.urlopen(req, timeout=5).read()).get("Answer"))
    except Exception:
        return None


def start_cloudflare_tunnel(port):
    import stat
    found = {"url": None, "registered": False, "ready": False}
    binary = "/tmp/cloudflared"
    try:
        if not os.path.exists(binary):
            urllib.request.urlretrieve(
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", binary)
            os.chmod(binary, os.stat(binary).st_mode | stat.S_IEXEC)
        tunnel = subprocess.Popen([binary, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate",
                                   "--protocol", "http2"],
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        import atexit
        atexit.register(tunnel.kill)
    except Exception as e:
        print(f"⚠️ Cloudflare tunnel unavailable: {e}")
        return found

    def _watch():
        for line in tunnel.stdout:  # keep draining so cloudflared never blocks on a full pipe
            match = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", line)
            if match and not found["url"]:
                found["url"] = match.group(0)
            if "Registered tunnel connection" in line:
                found["registered"] = True

    def _wait_ready():
        deadline, registered_at = time.time() + 300, None
        while time.time() < deadline:
            if found["registered"] and registered_at is None:
                registered_at = time.time()
            published = _dns_published(found["url"].split("//", 1)[1]) if found["url"] and registered_at else False
            if published or (published is None and time.time() - registered_at > 15):
                found["ready"] = True
                if PRINTED_LINKS.is_set():
                    print(f"\n✅ Cloudflare link is ready: {found['url']}\n")
                    sys.stdout.flush()
                return
            time.sleep(3)
        print("⚠️ The Cloudflare link did not come up within 5 minutes; use the Gradio share link.")

    threading.Thread(target=_watch, daemon=True).start()
    threading.Thread(target=_wait_ready, daemon=True).start()
    return found


PRINTED_LINKS = threading.Event()
print("🌐 Starting the public links (Cloudflare + Gradio share)...")
sys.stdout.flush()
cf = start_cloudflare_tunnel(SERVER_PORT)
demo.queue(max_size=20)
demo.launch(server_name="127.0.0.1", server_port=SERVER_PORT, share=True, inline=False, show_error=True,
            prevent_thread_lock=True, quiet=True, ssr_mode=False, show_api=False, max_threads=8,
            allowed_paths=[OUTPUT_DIR])
from fastapi.responses import FileResponse, JSONResponse
from fastapi.routing import APIRoute


def _progress_route():
    end = PROGRESS["finished"] or time.time()
    elapsed = end - PROGRESS["started"] if PROGRESS["started"] else 0.0
    return JSONResponse({**PROGRESS, "elapsed": elapsed}, headers={"Cache-Control": "no-store"})


def _audio_route():
    """The newest converted audio as a plain file (plays in the browser, with a download option): no Gradio involved."""
    path = _latest_audio()
    if not path:
        return JSONResponse({"error": "no audio yet"}, status_code=404)
    name = os.path.basename(path)
    return FileResponse(path, media_type="audio/wav",
                        headers={"Content-Disposition": f'inline; filename="{name}"', "Cache-Control": "no-store"})


for _path, _route in (("/aiq_progress", _progress_route), ("/aiq_audio", _audio_route)):
    demo.app.router.routes.insert(0, APIRoute(_path, _route, methods=["GET"]))  # ahead of Gradio's routes
for _ in range(60):  # give the tunnel up to ~30 s before printing the links
    if cf["ready"]:
        break
    time.sleep(0.5)

print("\n" + "=" * 70)
print("🎙️ SeedVC Voice Changer is LIVE!")
print("=" * 70)
if cf["ready"]:
    print(f"🌐 Cloudflare tunnel: {cf['url']}")
else:
    print(f"🌐 Cloudflare tunnel: ⏳ warming up{' (' + cf['url'] + ')' if cf['url'] else ''} - wait for the "
          f"'✅ Cloudflare link is ready' line before opening it")
print(f"🔗 Gradio share:      {getattr(demo, 'share_url', None) or '(unavailable, use the Cloudflare link)'}")
print(f"💾 Audio is saved to {OUTPUT_DIR} (Colab Files panel)")
print("🔊 Latest audio without the UI: <link>/aiq_audio")
print("=" * 70 + "\n")
PRINTED_LINKS.set()
sys.stdout.flush()
while True:  # keep the server process alive; conversion progress streams to the notebook cell
    time.sleep(3600)
'''

with open("/content/run_seedvc.py", "w", encoding="utf-8") as f:
    f.write(RUN_SCRIPT_CODE)
print("✅ run_seedvc.py written.")

if os.path.exists("/content/seedvc_settings.json"):  # Step 1's paths, for a Step 2 re-run after a session restart
    with open("/content/seedvc_settings.json") as f:
        for key, value in json.load(f).items():
            os.environ.setdefault(key, value)

# Library noise that is not an error; everything else (including tracebacks) is shown.
IGNORE = ("absl::InitializeLog", "Unable to register cu", "computation placer already registered", "MessageFactory",
          "GetPrototype", "Attempting to register factory", "UserWarning", "FutureWarning", "DeprecationWarning",
          "warnings.warn", "`torch_dtype` is deprecated", "TypedStorage is deprecated", "weight_norm")
env = dict(os.environ, PYTHONWARNINGS="ignore", TQDM_DISABLE="1", TRANSFORMERS_VERBOSITY="error",
           HF_HUB_DISABLE_PROGRESS_BARS="1", GRADIO_ANALYTICS_ENABLED="False", TOKENIZERS_PARALLELISM="false",
           PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")

print("🚀 Starting the SeedVC engine...\n")
import psutil


def stop_old_engine():
    # An engine still running from an earlier Step 2 (or left behind by a session restart) holds the GPU and RAM.
    stale = []
    for p in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            args, name = p.info["cmdline"] or [], (p.info["name"] or "").lower()
            engine = "python" in name and any(os.path.basename(a) == "run_seedvc.py" for a in args[1:3])
            if p.pid != os.getpid() and (engine or name == "cloudflared" or name.startswith("frpc")):
                p.kill()
                stale.append(p)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    psutil.wait_procs(stale, timeout=20)
    if stale:
        print(f"🧹 Stopped {len(stale)} engine process(es) from an earlier run: GPU memory and RAM are free again")


stop_old_engine()

# Step 1's downloads leave freed buffers in this kernel process: hand them back to the OS before the engine starts.
import ctypes
import gc
gc.collect()
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass


def _die_with_kernel():
    # Linux: the engine gets SIGKILL when this notebook kernel dies or restarts (no orphan holding the GPU).
    try:
        import signal
        ctypes.CDLL("libc.so.6").prctl(1, signal.SIGKILL)  # PR_SET_PDEATHSIG
    except Exception:
        pass


proc = subprocess.Popen([sys.executable, "-u", "/content/run_seedvc.py"], stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, env=env, cwd="/content", preexec_fn=_die_with_kernel)
import threading
_peak = {"ram": 0.0, "engine": 0.0}


def _watch_ram():  # peak RAM while the engine runs, reported if it dies
    engine = psutil.Process(proc.pid)
    while proc.poll() is None:
        try:
            vm = psutil.virtual_memory()
            _peak["ram"] = max(_peak["ram"], (vm.total - vm.available) / 1024**3)
            _peak["engine"] = max(_peak["engine"], engine.memory_info().rss / 1024**3)
        except Exception:
            pass
        time.sleep(0.5)


threading.Thread(target=_watch_ram, daemon=True).start()
# Read raw bytes so carriage returns survive: the engine redraws its live progress line in place, and a text-mode
# pipe would turn every redraw into a new line.
decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
pending = ""
try:
    while True:
        chunk = os.read(proc.stdout.fileno(), 65536)
        if not chunk:
            break
        pending += decoder.decode(chunk)
        parts = re.split(r"(?<=[\r\n])", pending)
        pending = parts.pop()  # text after the last line break / carriage return
        out = "".join(part for part in parts if not any(pattern in part for pattern in IGNORE))
        if pending and not any(pattern in pending for pattern in IGNORE):
            out, pending = out + pending, ""  # show the live progress line right away
        if out:
            sys.stdout.write(out)
            sys.stdout.flush()
    sys.stdout.write(pending)
    rc = proc.wait()
    if rc != 0:
        import signal
        how = f"killed by {signal.Signals(-rc).name}" if rc < 0 else f"exit code {rc}"
        hint = (" - the system killed it, which almost always means it ran out of RAM" if rc == -9 else
                " - a crash inside a native library (see the traceback above)" if rc in (-11, -6) else "")
        print(f"\n❌ The engine stopped ({how}){hint}.\n   Peak RAM while it ran: {_peak['ram']:.1f} GB in use of "
              f"{psutil.virtual_memory().total / 1024**3:.1f} GB (engine process {_peak['engine']:.1f} GB).")
except KeyboardInterrupt:
    proc.kill()  # frees the GPU and RAM right away
    proc.wait()
    print("🛑 Engine stopped. Re-run this cell to start it again.")

---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---